In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.ndimage import binary_closing, binary_dilation, label
import matplotlib.pyplot as plt


# ============================================================
# 1. CHARGEMENT DES SIGNAUX
# ============================================================
def charger_signaux(base_path, dossiers):
    """
    Charge les signaux JSON contenus dans plusieurs sous-dossiers et retourne
    un dictionnaire de DataFrames (un par dossier), chaque DataFrame ayant
    une colonne 'signal_brut' contenant un tableau NumPy centré par ligne.
    signal_brut = signal_original - moyenne(signal_original)
    """
    base_path = Path(base_path)
    dataset_dict = {}

    print("Début de l'extraction des signaux...")

    for dossier in dossiers:
        folder_path = base_path / dossier

        if not folder_path.exists():
            print(f"⚠️ Dossier introuvable : {dossier}")
            continue

        dict_mesures = {}

        for json_path in folder_path.glob("*.json"):
            try:
                with open(json_path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                idx = data["metadata"]["measurement_index"]
                samples_np = np.array(data["samples"], dtype=np.float64)

                # Centrage du signal dès le chargement
                samples_centre = samples_np - np.mean(samples_np)

                dict_mesures[idx] = samples_centre

            except Exception as e:
                print(f"❌ Erreur sur le fichier {json_path.name} : {e}")

        indices_tries = sorted(dict_mesures.keys())
        liste_tableaux = [dict_mesures[i] for i in indices_tries]

        df_dossier = pd.DataFrame(
            data={"signal_brut": liste_tableaux},
            index=indices_tries
        )

        dataset_dict[dossier] = df_dossier
        print(f"✅ Dossier '{dossier}' chargé : {df_dossier.shape[0]} lignes, {df_dossier.shape[1]} colonne.")

    return dataset_dict


# ============================================================
# 2. DÉTECTION DES ZONES PARASITES (inchangé)
# ============================================================
def detect_parasite_zones(signal, fs=20000, window_ms=5, smooth_ms=80,
                           k=3.7, min_duration_ms=20, margin_ms=15):
    signal = signal.astype(np.float64)
    n = len(signal)

    # 0. Retrait de l'offset DC
    offset = np.median(signal)
    signal_ac = signal - offset

    # 1. RMS fin
    window_size = max(1, int(fs * window_ms / 1000))
    kernel_fin = np.ones(window_size) / window_size
    pad = window_size // 2
    signal_padded = np.pad(signal_ac, pad, mode="reflect")
    rms_fin = np.sqrt(np.convolve(signal_padded**2, kernel_fin, mode="same"))
    rms_fin = rms_fin[pad:pad + n]

    # 2. RMS lissé
    smooth_size = max(1, int(fs * smooth_ms / 1000))
    kernel_smooth = np.ones(smooth_size) / smooth_size
    pad_s = smooth_size // 2
    rms_padded = np.pad(rms_fin, pad_s, mode="reflect")
    rms_smooth = np.convolve(rms_padded, kernel_smooth, mode="same")
    rms_smooth = rms_smooth[pad_s:pad_s + n]

    # 3. Seuil robuste
    med = np.median(rms_smooth)
    mad = np.median(np.abs(rms_smooth - med))
    threshold = med + k * mad * 1.4826

    mask = rms_smooth > threshold

    # 4. Nettoyage morphologique
    struct_close = np.ones(int(fs * 10 / 1000))
    mask = binary_closing(mask, structure=struct_close)

    struct_dilate = np.ones(int(fs * margin_ms / 1000))
    mask = binary_dilation(mask, structure=struct_dilate)

    labeled, n_zones = label(mask)
    zones = []
    min_duration_samples = int(fs * min_duration_ms / 1000)
    for i in range(1, n_zones + 1):
        idx = np.where(labeled == i)[0]
        if len(idx) >= min_duration_samples:
            zones.append((idx[0], idx[-1]))

    return zones, rms_smooth, threshold


def plot_detection(signal, zones, rms_local, threshold, title=""):
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(signal, color="steelblue", linewidth=0.5)
    for start, end in zones:
        axes[0].axvspan(start, end, color="red", alpha=0.3)
    axes[0].set_title(f"Signal brut — {title}")

    axes[1].plot(rms_local, color="darkorange", linewidth=0.8)
    axes[1].axhline(threshold, color="red", linestyle="--", label="seuil")
    axes[1].set_title("RMS glissant + seuil")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


# ============================================================
# 3. SUPPRESSION DES ZONES PARASITES (tes fonctions, verbatim)
# ============================================================
def remove_parasite_zones(signal, zones):
    """
    Supprime les zones de parasite d'un signal et recolle les morceaux restants.
    zones : liste de tuples (start_idx, end_idx) à supprimer (bornes incluses)
    """
    if not zones:
        return signal.copy()

    # On trie les zones par position de départ, par sécurité
    zones_triees = sorted(zones, key=lambda z: z[0])

    morceaux = []
    curseur = 0

    for start, end in zones_triees:
        # On garde le morceau AVANT le parasite (entre le curseur et le début de la zone)
        if start > curseur:
            morceaux.append(signal[curseur:start])
        # On avance le curseur juste APRÈS la fin du parasite
        curseur = end + 1

    # On ajoute ce qui reste après la dernière zone supprimée
    if curseur < len(signal):
        morceaux.append(signal[curseur:])

    return np.concatenate(morceaux) if morceaux else np.array([], dtype=signal.dtype)


def clean_dataframe(df, fs=20000, k=6, min_duration_ms=20, margin_ms=10, verbose=True):
    """
    Applique la détection + suppression de parasites à toutes les lignes d'un DataFrame.
    Retourne un nouveau DataFrame avec une colonne 'signal_clean' et 'nb_zones_supprimees'.
    """
    signaux_clean = []
    nb_zones_list = []
    longueurs_finales = []

    for idx, row in df.iterrows():
        signal = row["signal_brut"]
        zones, _, _ = detect_parasite_zones(
            signal, fs=fs, k=k, min_duration_ms=min_duration_ms, margin_ms=margin_ms
        )
        signal_clean = remove_parasite_zones(signal, zones)

        signaux_clean.append(signal_clean)
        nb_zones_list.append(len(zones))
        longueurs_finales.append(len(signal_clean))

        if verbose:
            print(f"Ligne {idx} : {len(zones)} zone(s) supprimée(s), "
                  f"taille {len(signal)} -> {len(signal_clean)}")

    df_clean = df.copy()
    df_clean["signal_clean"] = signaux_clean
    df_clean["nb_zones_supprimees"] = nb_zones_list
    df_clean["longueur_finale"] = longueurs_finales

    return df_clean


# ============================================================
# 4. PIPELINE COMPLET
# ============================================================
def pipeline_complet(base_path, dossiers, fs=20000, k=3, min_duration_ms=20, margin_ms=10, verbose=True):
    """
    Pipeline complet :
    1. Charge tous les signaux JSON (charger_signaux).
    2. Détecte + supprime les zones parasites de chaque DataFrame (clean_dataframe).

    Retourne un dict {dossier: DataFrame} où chaque DataFrame contient :
    - 'signal_brut', 'signal_clean', 'nb_zones_supprimees', 'longueur_finale'
    """
    # Étape 1 : chargement
    dataset_dict = charger_signaux(base_path, dossiers)

    print("\nDétection et suppression des parasites...")

    dataset_clean = {}
    for dossier, df in dataset_dict.items():
        df_clean = clean_dataframe(
            df, fs=fs, k=k, min_duration_ms=min_duration_ms,
            margin_ms=margin_ms, verbose=verbose
        )
        dataset_clean[dossier] = df_clean
        print(f"✅ {dossier} : nettoyage terminé ({df_clean.shape[0]} lignes).")

    return dataset_clean



# ============================================================
# Exemple d'utilisation
# ============================================================
if __name__ == "__main__":
    BASE_PATH = "nouvel_value/pompes"
    DOSSIERS = ["pompe_allumee","pompe_avec_fuite"]

    dataset_clean = pipeline_complet(BASE_PATH, DOSSIERS, fs=20000, k=3, min_duration_ms=20, margin_ms=10, verbose=True)

Début de l'extraction des signaux...
✅ Dossier 'pompe_allumee' chargé : 30 lignes, 1 colonne.
✅ Dossier 'pompe_avec_fuite' chargé : 30 lignes, 1 colonne.

Détection et suppression des parasites...
Ligne 1 : 2 zone(s) supprimée(s), taille 20000 -> 14317
Ligne 2 : 2 zone(s) supprimée(s), taille 20000 -> 14020
Ligne 3 : 2 zone(s) supprimée(s), taille 20000 -> 12908
Ligne 4 : 3 zone(s) supprimée(s), taille 20000 -> 12238
Ligne 5 : 1 zone(s) supprimée(s), taille 20000 -> 16627
Ligne 6 : 1 zone(s) supprimée(s), taille 20000 -> 16146
Ligne 7 : 3 zone(s) supprimée(s), taille 20000 -> 14023
Ligne 8 : 1 zone(s) supprimée(s), taille 20000 -> 16675
Ligne 9 : 1 zone(s) supprimée(s), taille 20000 -> 16668
Ligne 10 : 1 zone(s) supprimée(s), taille 20000 -> 17946
Ligne 11 : 2 zone(s) supprimée(s), taille 20000 -> 14505
Ligne 12 : 3 zone(s) supprimée(s), taille 20000 -> 14377
Ligne 13 : 3 zone(s) supprimée(s), taille 20000 -> 13927
Ligne 14 : 3 zone(s) supprimée(s), taille 20000 -> 13278
Ligne 15 : 2 z

In [2]:
import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import welch
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
 
# Bandes de fréquences pour le Welch (jusqu'à Nyquist = fs/2 = 10000 Hz pour fs=20000)
BANDES_HZ = [(0, 500), (500, 1000), (1000, 2000), (2000, 4000), (4000, 7000), (7000, 10000)]
 
 
# ============================================================
# 1. EXTRACTION DES FEATURES (simple)
# ============================================================
def extract_features(signal, fs=20000):
    """
    Calcule un petit vecteur de features à partir d'un signal_clean :
    - temporelles : RMS, variance, crest factor, kurtosis, peak-to-peak
    - fréquentielles : énergie par bande (Welch), sur BANDES_HZ
    """
    signal = signal.astype(np.float64)
    rms = np.sqrt(np.mean(signal ** 2))
    peak = np.max(np.abs(signal))
 
    features = {
        "rms": rms,
        "variance": np.var(signal),
        "crest_factor": peak / rms if rms > 0 else 0.0,
        "kurtosis": stats.kurtosis(signal),
        "peak_to_peak": np.ptp(signal),
    }
 
    # --- Énergie par bande de fréquence (Welch) ---
    freqs, psd = welch(signal, fs=fs, nperseg=min(4096, len(signal)))
    for f_min, f_max in BANDES_HZ:
        mask = (freqs >= f_min) & (freqs < f_max)
        energie = np.trapz(psd[mask], freqs[mask]) if mask.any() else 0.0
        features[f"energie_{f_min}_{f_max}hz"] = energie
 
    return features
 
 
def build_feature_dataframe(dataset_clean, fs=20000):
    """
    Applique extract_features à chaque signal_clean de pompe_allumee et
    pompe_avec_fuite (les autres dossiers éventuels sont ignorés).
    Retourne un DataFrame avec une colonne 'classe'.
    """
    frames = []
    for dossier in ("pompe_allumee", "pompe_avec_fuite"):
        if dossier not in dataset_clean:
            print(f"⚠️ Dossier '{dossier}' absent de dataset_clean, ignoré.")
            continue
        for idx, row in dataset_clean[dossier].iterrows():
            feats = extract_features(row["signal_clean"], fs=fs)
            feats["classe"] = dossier
            frames.append(feats)
 
    df_features = pd.DataFrame(frames).reset_index(drop=True)
    print(f"✅ Features calculées : {df_features.shape[0]} lignes.")
    return df_features
 
 
# ============================================================
# 2. ENTRAÎNEMENT + ÉVALUATION
# ============================================================
def pipeline_isolation_forest(dataset_clean, fs=20000, test_size=0.1, random_state=42):
    """
    1. Extraction des features (pompe_allumee + pompe_avec_fuite uniquement).
    2. Split pompe_allumee : 90% train / 10% test.
    3. Entraînement de l'Isolation Forest (paramètres classiques) sur
       les 90% de pompe_allumee.
    4. Test sur les 10% restants de pompe_allumee + 100% de pompe_avec_fuite.
    5. Métriques simples : matrice de confusion + classification report.
    """
    df_features = build_feature_dataframe(dataset_clean, fs=fs)
    feature_cols = [c for c in df_features.columns if c != "classe"]
 
    df_allumee = df_features[df_features["classe"] == "pompe_allumee"]
    df_fuite = df_features[df_features["classe"] == "pompe_avec_fuite"]
 
    df_train, df_test_allumee = train_test_split(
        df_allumee, test_size=test_size, random_state=random_state
    )
 
    # --- Standardisation (fit sur le train uniquement) ---
    scaler = StandardScaler()
    X_train = scaler.fit_transform(df_train[feature_cols])
 
    # --- Isolation Forest, paramètres classiques ---
    model = IsolationForest(n_estimators=100, random_state=random_state)
    model.fit(X_train)
 
    print(f"✅ Isolation Forest entraîné sur {len(df_train)} mesures 'pompe_allumee'.")
 
    # --- Jeu de test : 10% pompe_allumee restant + 100% pompe_avec_fuite ---
    df_test = pd.concat([df_test_allumee, df_fuite])
    X_test = scaler.transform(df_test[feature_cols])
 
    y_true = (df_test["classe"] == "pompe_avec_fuite").astype(int).values  # 1 = fuite
    y_pred_raw = model.predict(X_test)       # -1 = anomalie, 1 = normal (seuil par défaut)
    y_pred = (y_pred_raw == -1).astype(int)  # 1 = anomalie détectée
 
    print(f"\nJeu de test : {len(df_test_allumee)} pompe_allumee + {len(df_fuite)} pompe_avec_fuite")
 
    cm = confusion_matrix(y_true, y_pred)
    print("\n=== Matrice de confusion ===")
    print("                  Prédit normal   Prédit fuite")
    print(f"Vrai normal       {cm[0, 0]:<15} {cm[0, 1]:<15}")
    print(f"Vrai fuite        {cm[1, 0]:<15} {cm[1, 1]:<15}")
 
    print("\n=== Rapport de classification ===")
    print(classification_report(y_true, y_pred, target_names=["normal", "fuite"], zero_division=0))
 
    return {
        "model": model,
        "scaler": scaler,
        "feature_cols": feature_cols,
        "confusion_matrix": cm,
        "df_test": df_test.assign(prediction=y_pred, vrai_label=y_true),
    }
 
 
# ============================================================
# Exemple d'utilisation (à la suite de ton pipeline_complet existant)
# ============================================================
if __name__ == "__main__": 
    BASE_PATH = "nouvel_value/fuite_v2"
    DOSSIERS = ["pompe_allumee", "pompe_avec_fuite"]
 
    dataset_clean = pipeline_complet(
        BASE_PATH, DOSSIERS, fs=20000, k=3,
        min_duration_ms=20, margin_ms=10, verbose=False
    )
 
    resultats = pipeline_isolation_forest(dataset_clean, fs=20000)

Début de l'extraction des signaux...
✅ Dossier 'pompe_allumee' chargé : 30 lignes, 1 colonne.
✅ Dossier 'pompe_avec_fuite' chargé : 30 lignes, 1 colonne.

Détection et suppression des parasites...
✅ pompe_allumee : nettoyage terminé (30 lignes).
✅ pompe_avec_fuite : nettoyage terminé (30 lignes).
✅ Features calculées : 60 lignes.


C:\Users\natha\AppData\Local\Temp\ipykernel_14212\3826273556.py:39: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  energie = np.trapz(psd[mask], freqs[mask]) if mask.any() else 0.0


✅ Isolation Forest entraîné sur 27 mesures 'pompe_allumee'.

Jeu de test : 3 pompe_allumee + 30 pompe_avec_fuite

=== Matrice de confusion ===
                  Prédit normal   Prédit fuite
Vrai normal       2               1              
Vrai fuite        11              19             

=== Rapport de classification ===
              precision    recall  f1-score   support

      normal       0.15      0.67      0.25         3
       fuite       0.95      0.63      0.76        30

    accuracy                           0.64        33
   macro avg       0.55      0.65      0.51        33
weighted avg       0.88      0.64      0.71        33

